In [1]:
import re
import string
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
# We have imported the libraries that we will need.
# The Goal is to make a NLP model. We Will use the imdb reviews to train our model to predict whether a review is
# positive or negative.
# Let us start with reading the reviews and labels.
def load_number_of_reviews_and_labels(reviews_file, labels_file, number):
    with open(reviews_file, "r", encoding = "utf-8") as file:
        reviews = file.readlines()
    with open(labels_file, "r", encoding = "utf-8") as file:
        labels = file.readlines()
    reviews = [r for r in reviews[:number]]
    labels = [l for l in labels[:number]]
    return reviews, labels

In [3]:
# Now before we move on to preprocessing, we will take 100 reviews first and do the preprocessing on them before
# doing the work with all of the reviews.
reviews, labels = load_number_of_reviews_and_labels("imdb_review.txt", "imdb_labels.txt", 100)

In [6]:
# let's take a look at the first review:
print(reviews[0])

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [7]:
# and the next review:
print(reviews[1])

"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only ""has got all the polari"" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly well done

In [4]:
# We need to clean up the data which is preprocessing.
def remove_noise(text):
    text = text.lower() # convert to lowercase
    text = re.sub(r"\d+", " ", text) # remove numbers
    text = BeautifulSoup(text, "html.parser").get_text() # remove html tags
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text) # remove punctuation
    text = re.sub(r"\s+", " ", text).strip() # get rid of extra spaces on the edges and convert multipule spaces into one
    return text

# let's tokenize. meaning spliting text to words(tokens).
def tokenize(text):
    return nltk.word_tokenize(text)
    
# let's remove stopwords and lemmatize:
# removing stopwords means getting rid of meaningless words like 'and, the, so,...'.
def remove_stopwords(tokens):
    stop_words = set(stopwords.words("english"))
    return [word for word in tokens if word not in stop_words]

# lemmatize means reducing words to their dictionary form considering context. for example 'running -> run'
def lemmatize_tokens(tokens):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in tokens]

In [5]:
# now let's combine them into a pipeline:
def preprocessing_pipeline(text):
    text = remove_noise(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

In [6]:
def preprocess(text):
    return [preprocessing_pipeline(r) for r in reviews]

In [8]:
# now let's do the preprocessing for 100 reviews:
preprocessed_reviews = preprocess(reviews)

In [9]:
# let's see the first 2 preprocessed review:
print(preprocessed_reviews[:2])

['one reviewer mentioned watching oz episode hooked right exactly happened first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda em city home many aryan muslim gangsta latino christian italian irish scuffle death stare dodgy dealing shady agreement never far away would say main appeal show due fact go show dare forget pretty picture painted mainstream audience forget charm forget romance oz mess around first episode ever saw struck nasty surreal say ready watched developed taste oz got accustomed high level graphic violence violence injustice crooked guard sold nickel inmate kill order get away well mannered middle class inmate turned prison bitch due lack street skill prison experience watching oz 

In [14]:
# Bag of words:
from sklearn.feature_extraction.text import CountVectorizer
def bag_of_words(preprocessed_reviews):
    vectorizer = CountVectorizer()
    X_bow = vectorizer.fit_transform(preprocessed_reviews)
    return X_bow, vectorizer

In [15]:
# TF-IDF:
from sklearn.feature_extraction.text import TfidfVectorizer
def tfidf(preprocessed_reviews):
    vectorizer = TfidfVectorizer()
    X_tfidf = vectorizer.fit_transform(preprocessed_reviews)
    return X_tfidf, vectorizer

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(preprocessed_reviews, labels, test_size = 0.2, random_state = 42)

In [25]:
X_train_bow, bow_vectorizer = bag_of_words(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [26]:
X_train_tfidf, tfidf_vectorizer = tfidf(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [27]:
from sklearn.svm import LinearSVC
clf_bow = LinearSVC()
clf_bow.fit(X_train_bow, y_train)

clf_tfidf = LinearSVC()
clf_tfidf.fit(X_train_tfidf, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [28]:
y_pred_bow = clf_bow.predict(X_test_bow)

In [29]:
y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)

In [32]:
from sklearn.metrics import accuracy_score, classification_report
print("BoW Accuracy:", accuracy_score(y_test, y_pred_bow))
print(classification_report(y_test, y_pred_bow))

print ("TF-IDF Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print(classification_report(y_test, y_pred_tfidf))

BoW Accuracy: 0.5
              precision    recall  f1-score   support

   negative
       0.38      1.00      0.55         6
   positive
       1.00      0.29      0.44        14

    accuracy                           0.50        20
   macro avg       0.69      0.64      0.49        20
weighted avg       0.81      0.50      0.47        20

TF-IDF Accuracy: 0.4
              precision    recall  f1-score   support

   negative
       0.33      1.00      0.50         6
   positive
       1.00      0.14      0.25        14

    accuracy                           0.40        20
   macro avg       0.67      0.57      0.38        20
weighted avg       0.80      0.40      0.33        20



# References:
[geeksforgeeks_NLP](https://www.geeksforgeeks.org/nlp/natural-language-processing-overview/)

[geeksforgeeks_SVM](https://www.geeksforgeeks.org/machine-learning/support-vector-machine-algorithm/)

[geeksforgeeks_code](https://www.geeksforgeeks.org/nlp/sentiment-analysis-on-imdb-movie-reviews/)

[Kaggle_list_of_codes](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/code)

[Kaggle_1](https://www.kaggle.com/code/yaninakostiv/imdb-sentiment-analysis-with-tf-idf-lr-nb#TF-IDF-+-LogReg-/-NaiveBayes)

[Medium](https://bellouchelhassan.medium.com/easy-step-by-step-tutorial-for-a-sentiment-analysis-project-of-imdb-movie-reviews-a9e443f4f493)

[NLTK](https://www.nltk.org/book/ch01.html)

[Bag of words/TF-IDF](https://www.geeksforgeeks.org/nlp/bag-of-words-vs-tf-idf/)